In [1]:
import os 
import pickle
import numpy as np
from qkeras.utils import load_qmodel
#from keras.models import load_model
import hls4ml
from sklearn.metrics import accuracy_score
from hls4ml.model.profiling import numerical
import hls4ml.report as report
import tensorflow as tf
from hls4ml.backends import get_backend
from qkeras import QDense, QConv2D

2026-10-23 18:00:46.967823: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-23 18:00:46.991576: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-23 18:00:47.145476: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-23 18:00:47.146663: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-23 18:00:47.635684: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:3

In [5]:
def load_pickle_data(path):
    print(f"Lendo {path} ...")
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"], data["y"]

def trace_layers(config):
    config['Model']['TraceOutput'] = True
    for layer in config['LayerName']:
        config['LayerName'][layer]['Trace'] = True


def print_dict(d, indent=0):
    for key, value in d.items():
        print('  ' * indent + str(key), end='')
        if isinstance(value, dict):
            print()
            print_dict(value, indent + 1)
        else:
            print(':' + ' ' * (20 - len(key) - 2 * indent) + str(value))


def get_layer_rf_profiles(
    qmodel,
    backend_name,
    fpga_part,
    io_type,
    clock_period_ns,
    strategy,
    base_dir,
):
    backend_obj = hls4ml.backends.get_backend(backend_name)
    profiles = {"min": {}, "mid": {}, "max": {}}
    report = []

    probe_config = hls4ml.utils.config_from_keras_model(
        qmodel,
        granularity="name",
        backend=backend_name,
        default_reuse_factor=1,
    )
    if "output_dense" in probe_config.get("LayerName", {}):
        probe_config["LayerName"]["output_dense"]["Precision"]["result"] = (
            "fixed<16,6,RND,SAT>"
        )
    probe_config["Model"]["Strategy"] = strategy

    clock_str = str(clock_period_ns).replace(".", "p")
    probe_dir = os.path.join(
        ".", "t1_projects", "_rf_profile_probe",
        f"{base_dir}_{io_type}_{strategy}_cp{clock_str}",
    )
    probe_model = hls4ml.converters.convert_from_keras_model(
        qmodel,
        hls_config=probe_config,
        output_dir=probe_dir,
        part=fpga_part,
        clock_period=clock_period_ns,
        io_type=io_type,
        backend=backend_name,
    )

    for layer in probe_model.get_layers():
        try:
            n_in, n_out = backend_obj.get_layer_mult_size(layer)
        except (Exception,):
            continue

        valid_rf = backend_obj.get_valid_reuse_factors(
            n_in=int(n_in),
            n_out=int(n_out),
        )
        if not valid_rf:
            print(
                f"[AVISO] Nenhum RF válido retornado para "
                f"{layer.name} ({layer.class_name}); camada ignorada."
            )
            continue

        min_rf = valid_rf[0]
        max_rf = valid_rf[-1]
        midpoint = (min_rf + max_rf) / 2
        mid_rf = min(valid_rf, key=lambda rf: (abs(rf - midpoint), rf))

        profiles["min"][layer.name] = min_rf
        profiles["mid"][layer.name] = mid_rf
        profiles["max"][layer.name] = max_rf

        report.append({
            "name": layer.name,
            "type": layer.class_name,
            "n_in": int(n_in),
            "n_out": int(n_out),
            "min_rf": min_rf,
            "mid_rf": mid_rf,
            "max_rf": max_rf,
            "n_valid": len(valid_rf),
            "valid_rf": valid_rf,
        })

    print("\nRFs calculados por camada (dimensões internas do hls4ml):")
    for row in report:
        print(
            f"  {row['name']} ({row['type']}): "
            f"n_in={row['n_in']}, n_out={row['n_out']}, "
            f"min={row['min_rf']}, mid={row['mid_rf']}, "
            f"max={row['max_rf']} ({row['n_valid']} RFs válidos), "
            f"valids={row['valid_rf']}"
        )

    if not report:
        raise RuntimeError(
            "Nenhuma camada Dense/Conv compatível foi encontrada no ModelGraph "
            "temporário; não foi possível calcular os perfis de RF."
        )

    return profiles

In [3]:
TRAIN_PICKLE_PATH = "./dataset/sat6_train.pkl"
TEST_PICKLE_PATH = "./dataset/sat6_test.pkl"

X_train_all, y_train_all = load_pickle_data(TRAIN_PICKLE_PATH)
X_test, y_test = load_pickle_data(TEST_PICKLE_PATH)

X_train_all = X_train_all.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

val_idx = int(0.75 * len(X_train_all))
X_train, y_train = X_train_all[:val_idx], y_train_all[:val_idx]
X_val, y_val = X_train_all[val_idx:], y_train_all[val_idx:]

n_classes = 6
input_shape = X_train.shape[1:]
train_size = len(X_train)


Lendo ./dataset/sat6_train.pkl ...
Lendo ./dataset/sat6_test.pkl ...


In [9]:
bits= 4
BASE_DIR = f't1_{bits}'
MODEL_NAME = 'model' 
FINAL_PATH = f"models/{BASE_DIR}/{MODEL_NAME}.h5"
STRATEGY = 'Resource' # 'Latency', 'Resource'
IO_TYPE = "io_stream" # "io_parallel", 'io_stream'
CLOCK_PERIOD_NS = 10
FPGA_PART = "xc7z020clg400-1"  
BACKEND = "Vitis"


qmodel = load_qmodel(FINAL_PATH)
profiles = get_layer_rf_profiles(qmodel, BACKEND, FPGA_PART, IO_TYPE, CLOCK_PERIOD_NS, STRATEGY, BASE_DIR)
profiles


RFs calculados por camada (dimensões internas do hls4ml):
  conv_0 (Conv2D): n_in=36, n_out=20, min=1, mid=360, max=720 (14 RFs válidos), valids=[1, 2, 3, 4, 6, 9, 12, 18, 36, 72, 144, 180, 360, 720]
  conv_1 (Conv2D): n_in=180, n_out=8, min=1, mid=720, max=1440 (21 RFs válidos), valids=[1, 2, 3, 4, 5, 6, 9, 10, 12, 15, 18, 20, 30, 36, 45, 60, 90, 180, 360, 720, 1440]
  output_dense (Dense): n_in=6272, n_out=6, min=1, mid=18816, max=37632 (27 RFs válidos), valids=[1, 2, 4, 7, 8, 14, 16, 28, 32, 49, 56, 64, 98, 112, 128, 196, 224, 392, 448, 784, 896, 1568, 3136, 6272, 12544, 18816, 37632]


{'min': {'conv_0': 1, 'conv_1': 1, 'output_dense': 1},
 'mid': {'conv_0': 360, 'conv_1': 720, 'output_dense': 18816},
 'max': {'conv_0': 720, 'conv_1': 1440, 'output_dense': 37632}}

In [ ]:
REUSE_FACTOR  =  1
if CLOCK_PERIOD_NS != 10:
    OUTPUT_DIR = f"/project_hlsm/prj_{BASE_DIR}_{IO_TYPE}_rf{REUSE_FACTOR * 100:.0f}_{STRATEGY}_cp{CLOCK_PERIOD_NS}"
else:
    OUTPUT_DIR = f"/project_hlsm/prj_{BASE_DIR}_{IO_TYPE}_rf{REUSE_FACTOR * 100:.0f}_{STRATEGY}_fifo100k_100BRAM"


hls_config = hls4ml.utils.config_from_keras_model(
    qmodel,
    granularity="name",
    backend='Vitis',
    #default_reuse_factor=REUSE_FACTOR,
)

hls_config['LayerName']['output_dense']['Precision']['result'] = 'fixed<16,6,RND,SAT>'
hls_config['Model']['Strategy'] = STRATEGY

#for name, rf_max in layers_rf.items():
#    rf = round(1 + REUSE_FACTOR * (rf_max - 1))
#    hls_config["LayerName"][name]["ReuseFactor"] = rf

trace_layers(hls_config)
print_dict(hls_config)

hls_model = hls4ml.converters.convert_from_keras_model(
    qmodel,
    hls_config=hls_config,
    output_dir=OUTPUT_DIR,
    part=FPGA_PART,
    clock_period=CLOCK_PERIOD_NS,
    io_type=IO_TYPE,
    backend=BACKEND,
)
#hls_model.compile()

In [ ]:
#hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True, to_file=None)

In [ ]:
#numerical(model=qmodel, hls_model=hls_model, X=X_test[:2000], plot='boxplot')

In [ ]:
X_test_reduced = np.ascontiguousarray(X_test[:3000])
Y_test_reduced = np.array(y_test[:3000])

y_qkeras = qmodel.predict(X_test_reduced)
y_pred_hls = hls_model.predict(X_test_reduced)

accuracy_qkeras = accuracy_score(
    np.argmax(Y_test_reduced, axis=1),
    np.argmax(y_qkeras, axis=1)
)

accuracy_hls = accuracy_score(
    np.argmax(Y_test_reduced, axis=1),
    np.argmax(y_pred_hls, axis=1)
)

print('Accuracy base: {}'.format(accuracy_qkeras))
print('Accuracy after hls4ml: {}'.format(accuracy_hls))

accuracy_file = os.path.join(OUTPUT_DIR, 'accuracy.txt')

with open(accuracy_file, 'w') as f:
    f.write(f'Accuracy base: {accuracy_qkeras:.6f}\n')
    f.write(f'Accuracy after hls4ml: {accuracy_hls:.6f}\n')

In [ ]:
hls_model.build(csim=True, synth=True, vsynth=True, cosim=True)

In [ ]:
report.read_vivado_report(hls_model.config.get_output_dir())